<a href="https://colab.research.google.com/github/gabrielbecmar1932/Ingenieria-Logistica/blob/main/Practica_Ing_Logistica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed CodecBzip2 ───────── v0.8.5
   Installed MutableArithmetics ─ v1.8.0
   Installed MathOptInterface ─── v1.51.1
   Installed JuMP ─────────────── v1.30.1
    Updating `~/.julia/environments/v1.12/Project.toml`
  [4076af6c] + JuMP v1.30.1
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [523fee87] + CodecBzip2 v0.8.5
  [4076af6c] + JuMP v1.30.1
  [b8f27783] + MathOptInterface v1.51.1
  [d8a4904e] + MutableArithmetics v1.8.0
Precompiling packages...
   8016.7 ms  ✓ CodecBzip2
  18501.1 ms  ✓ MutableArithmetics
  95391.8 ms  ✓ MathOptInterface
  65709.5 ms  ✓ JuMP
  4 dependencies successfully precompiled in 187 seconds. 480 already precompiled.
   Resolving package versions...
   Installed OpenBLAS32_jll ─ v0.3.33+1
   Installed MathOptIIS ───── v0.2.0
   Installed HiGHS_jll ────── v1.14.0+0
   Installed HiGHS ────────── v1.23.0
  Installing 2 artifacts
   Installed artifact

In [ ]:
using JuMP, HiGHS

# Problemas Clasicos

## Problemas de asignacion

### **Asignacion 2D**

Dados n trabajadores y n tareas con coste c[i,j] de asignar el trabajador i a la tarea j.
Objetivo: minimizar el coste total de asignación.
Variables: x[i,j] ∈ {0,1} — 1 si trabajador i hace tarea j.
Restricciones:
- Cada trabajador hace exactamente una tarea: sum_j x[i,j] = 1 ∀i
- Cada tarea la hace exactamente un trabajador: sum_i x[i,j] = 1 ∀j

In [ ]:
# Datos
c = [5 3 9 1;
     8 7 2 6;
     4 2 7 3;
     7 9 4 5]

n = 4  # número de trabajadores y tareas

# Crear el modelo
model = Model(HiGHS.Optimizer)

# Variables de decisión: matriz 3x3 de binarias
@variable(model, x[1:n, 1:n], Bin)

# Función objetivo: minimizar coste total
@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n))

# Restricciones: cada trabajador hace una tarea
@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n) == 1)

# Restricciones: cada tarea la hace un trabajador
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n) == 1)

# Resolver
optimize!(model)

# Mostrar resultado
println("Coste mínimo: ", objective_value(model))
println("Asignación:")
for i in 1:n, j in 1:n
    if value(x[i,j]) > 0.5
        println("  Trabajador $i -> Tarea $j")
    end
end

Running HiGHS 1.14.0 (git hash: 7df0786de3): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline 
MIP has 8 rows; 16 cols; 32 nonzeros; 16 integer variables (16 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 9e+00]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+00]
Presolving model
8 rows, 16 cols, 32 nonzeros 0s
8 rows, 16 cols, 32 nonzeros 0s
Presolve reductions: rows 8(-0); columns 16(-0); nonzeros 32(-0) - Not reduced
Objective function is integral with scale 1

Solving MIP model with:
   8 rows
   16 cols (16 binary, 0 integer, 0 implied int., 0 continuous, 0 domain fixed)
   32 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial

### **Asignacion 3D**

Añade una tercera dimensión (turno k) al problema 2D.
Variables: x[i,j,k] ∈ {0,1} — 1 si trabajador i hace tarea j en turno k.
Restricciones (versión axial — fija un índice, suma los otros dos):
- sum_j sum_k x[i,j,k] = 1 ∀i
- sum_i sum_k x[i,j,k] = 1 ∀j
- sum_i sum_j x[i,j,k] = 1 ∀k
Versión planar: fija dos índices y suma uno → más restrictiva.

In [ ]:
# Datos
c = rand(1:10, 3, 3, 3)
n = 3  # número de trabajadores, tareas y turnos

# Crear el modelo
model = Model(HiGHS.Optimizer)

# Variables de decisión: matriz 3x3 de binarias
@variable(model, x[1:n, 1:n, 1:n], Bin)

# Función objetivo: minimizar coste total
@objective(model, Min, sum(c[i,j, k] * x[i,j,k] for i in 1:n, j in 1:n, k in 1:n))

# Restricciones: cada trabajador hace una tarea
@constraint(model, [i in 1:n], sum(x[i,j,k] for j in 1:n, k in 1:n) == 1)

# Restricciones: cada tarea la hace un trabajador
@constraint(model, [j in 1:n], sum(x[i,j,k] for i in 1:n, k in 1:n) == 1)

# Restricciones: cada tarea la hace un trabajador
@constraint(model, [k in 1:n], sum(x[i,j,k] for i in 1:n, j in 1:n) == 1)

# Resolver
optimize!(model)

# Mostrar resultado
println("Coste mínimo: ", objective_value(model))
println("Asignación:")
for i in 1:n, j in 1:n, k in 1:n
    if value(x[i,j,k]) > 0.5
        println("  Trabajador $i -> Tarea $j -> Turno$k")
    end
end

Running HiGHS 1.14.0 (git hash: 7df0786de3): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline 
MIP has 9 rows; 27 cols; 81 nonzeros; 27 integer variables (27 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 1e+01]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+00]
Presolving model
9 rows, 27 cols, 81 nonzeros 0s
9 rows, 27 cols, 81 nonzeros 0s
Presolve reductions: rows 9(-0); columns 27(-0); nonzeros 81(-0) - Not reduced
Objective function is integral with scale 1

Solving MIP model with:
   9 rows
   27 cols (27 binary, 0 integer, 0 implied int., 0 continuous, 0 domain fixed)
   81 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial

## Problema de Transporte


Dados orígenes I con stock s[i] y destinos J con demanda d[j].
Objetivo: minimizar coste total de transporte.
Variables: x[i,j] >= 0 — unidades enviadas de fábrica i a cliente j.
Restricciones:
- Stock: sum_j x[i,j] <= s[i] ∀i
- Demanda: sum_i x[i,j] == d[j] ∀j
Extensión multiproducto: añade índice k → variables trans[i,j,k].

In [ ]:
# Datos
s = [30, 40, 30]        # stock de cada fábrica
d = [25, 35, 40]        # demanda de cada cliente
c = [2 3 1;
     5 4 8;
     5 6 8]

n = 3  # fábricas
m = 3  # clientes

model = Model(HiGHS.Optimizer)

# Variables (continuas, no Bin)
@variable(model, x[1:n, 1:m] >= 0)

# Objetivo
@objective(model, Min, sum(c .* x))

# Restricción stock
@constraint(model, p[i=1:n], sum(x[i, :]) <= s[i])

# Restricción demanda
@constraint(model, r[j=1:m], sum(x[:, j]) == d[j])

optimize!(model)
println("Coste mínimo: ", objective_value(model))
println("Termination status: ", termination_status(model))
println("Envíos:")
for i in 1:n, j in 1:m
    if value(x[i,j]) > 0
        println("  Fábrica $i -> Cliente $j : $(value(x[i,j])) unidades")
    end
end

Running HiGHS 1.14.0 (git hash: 7df0786de3): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline 
LP has 6 rows; 9 cols; 18 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 8e+00]
  Bound   [0e+00, 0e+00]
  RHS     [2e+01, 4e+01]
Presolving model
6 rows, 9 cols, 18 nonzeros 0s
Dependent equations search running on 3 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
6 rows, 9 cols, 18 nonzeros 0s
Presolve reductions: rows 6(-0); columns 9(-0); nonzeros 18(-0) - Not reduced
Problem not reduced by presolve: solving the LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 3(100) 0.0s
          7     3.7500000000e+02 Pr: 0(0) 0.0s

Model status        : Optimal
Simplex   iterations: 7
Objective value     :  3.7500000000e+02
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00
Cos

## Problema de Flujo Máximo


Red con nodos y aristas (origen, destino, capacidad).
Objetivo: maximizar flujo total desde nodo origen hasta nodo destino.
Variables: x[a] >= 0 — flujo por cada arista a.
Restricciones:
- Capacidad: x[a] <= a[3] ∀a
- Conservación: flujo que entra = flujo que sale en nodos intermedios
Teorema max-flow min-cut: flujo máximo = capacidad del corte mínimo.

Necesitamos representar la red. En optimización de redes se usan aristas en lugar de matrices, porque no todos los nodos están conectados entre sí.
El grafo del ejemplo:
```
1 --[4]--> 2 --[2]--> 4
|                     ^
[3]                   |
|                     |
v                     |
3 ------[3]---------->
```
En código representamos las aristas como una lista de tuplas (origen, destino, capacidad):

julia# (origen, destino, capacidad)

```
aristas = [(1,2,4), (1,3,3), (2,4,2), (3,4,3)]

origen = 1
destino = 4
nodos = 1:4
```




In [ ]:
aristas = [(1,2,4), (1,3,3), (2,4,2), (3,4,3)]
origen = 1
destino = 4
nodos = 1:4

model = Model(HiGHS.Optimizer)
set_silent(model)
@variable(model, x[aristas] >= 0)
@objective(model, Max, sum(x[a] for a in aristas if a[1] == origen))
@constraint(model, [a in aristas], x[a] <= a[3])
@constraint(model, [v in nodos; v != origen && v != destino],
    sum(x[a] for a in aristas if a[2] == v) ==
    sum(x[a] for a in aristas if a[1] == v))

optimize!(model)
println("Flujo máximo: ", objective_value(model))
for a in aristas
    println("  $(a[1]) -> $(a[2]) : $(value(x[a])) / $(a[3])")
end

Flujo máximo: 5.0
  1 -> 2 : 2.0 / 4
  1 -> 3 : 3.0 / 3
  2 -> 4 : 2.0 / 2
  3 -> 4 : 3.0 / 3


## Flujo Maximo con coste minimo



Combina transporte y flujo máximo. Aristas con capacidad Y coste.
Flujo total requerido está fijado, se minimiza el coste.
Variables: x[a] >= 0 — flujo por cada arista a.
Restricciones:
- Flujo total: sum x[a] == flujo_requerido (aristas que salen del origen)
- Capacidad: x[a] <= a[3] ∀a
- Conservación: igual que flujo máximo

Ejemplo: misma red pero con costes en cada arista y un flujo fijo que enviar de 1 a 4:


```
1 --[coste 2, cap 4]--> 2 --[coste 3, cap 2]--> 4
|                                                ^
[coste 1, cap 3]                                 |
|                                                |
v                                                |
3 ----------[coste 5, cap 3]------------------->
```
Queremos enviar 4 unidades de 1 a 4 al mínimo coste.
Las aristas ahora tienen 4 datos: (origen, destino, capacidad, coste):
```
juliaaristas = [(1,2,4,2), (1,3,3,1), (2,4,2,3), (3,4,3,5)]
origen = 1
destino = 4
flujo_requerido = 4
nodos = 1:4
```

In [ ]:
aristas = [(1,2,4,2), (1,3,3,1), (2,4,2,3), (3,4,3,5)]
origen = 1
destino = 4
flujo_requerido = 3
nodos = 1:4

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[aristas] >= 0)

@objective(model, Min, sum(a[4] * x[a] for a in aristas))

@constraint(model, sum(x[a] for a in aristas if a[1] == origen) == flujo_requerido)

@constraint(model, [a in aristas], x[a] <= a[3])

@constraint(model, [v in nodos; v != origen && v != destino],
    sum(x[a] for a in aristas if a[2] == v) ==
    sum(x[a] for a in aristas if a[1] == v))

optimize!(model)
println("Coste mínimo: ", objective_value(model))
for a in aristas
    println("  $(a[1]) -> $(a[2]) : $(value(x[a])) / $(a[3]) (coste $(a[4]))")
end

Coste mínimo: 16.0
  1 -> 2 : 2.0 / 4 (coste 2)
  1 -> 3 : 1.0 / 3 (coste 1)
  2 -> 4 : 2.0 / 2 (coste 3)
  3 -> 4 : 1.0 / 3 (coste 5)


### ÁRBOL GENERADOR MÍNIMO (Minimum Spanning Tree)



Dado un grafo con nodos y aristas con coste, encuentra el conjunto de aristas
que conecta todos los nodos con el mínimo coste total sin formar ciclos.

Es una variante del problema de flujo a coste mínimo del tema 1: el nodo 1
actúa como fuente que distribuye n-1 unidades, una para cada nodo destino.
La conectividad queda garantizada por el flujo auxiliar, que solo puede
circular por aristas seleccionadas.

Variables:
  - x[i,j] ∈ {0,1}  → 1 si incluimos la arista (i,j), solo con i < j
                     para evitar duplicados al ser el grafo no dirigido
  - f[i,j] >= 0      → flujo auxiliar que garantiza conectividad

Restricciones:
  - Exactamente n-1 aristas (un árbol de n nodos tiene siempre n-1 aristas)
  - El flujo solo circula por aristas seleccionadas, en cualquier dirección
  - Nodo 1 genera n-1 unidades de flujo, una por cada nodo a conectar
  - Cada nodo recibe exactamente 1 unidad, garantizando que todos
    están conectados al nodo 1

In [ ]:
using JuMP, HiGHS

n = 5
c = [0 3 1 5 8;
     3 0 6 7 9;
     1 6 0 4 2;
     5 7 4 0 3;
     8 9 2 3 0]

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[i in 1:n, j in 1:n; i < j], Bin)
@variable(model, f[1:n, 1:n] >= 0)

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i < j))

# Exactamente n-1 aristas
@constraint(model, sum(x[i,j] for i in 1:n, j in 1:n if i < j) == n-1)

# Flujo solo por aristas seleccionadas (en ambas direcciones)
@constraint(model, [i in 1:n, j in 1:n; i < j], f[i,j] + f[j,i] <= (n-1) * x[i,j])

# Nodo 1 genera n-1 unidades
@constraint(model, sum(f[1,j] for j in 2:n) == n-1)

# Cada nodo recibe exactamente 1 unidad
@constraint(model, [i in 2:n],
    sum(f[j,i] for j in 1:n if j != i) - sum(f[i,j] for j in 1:n if j != i) == 1)

optimize!(model)

println("Coste mínimo: ", objective_value(model))
for i in 1:n, j in (i+1):n
    if value(x[i,j]) > 0.5
        println("  $i — $j  (coste=$(c[i,j]))")
    end
end

Coste mínimo: 9.0
  1 — 2  (coste=3)
  1 — 3  (coste=1)
  3 — 5  (coste=2)
  4 — 5  (coste=3)


#Problemas de Logistica en Transporte

 ##  SHORTEST PATH PROBLEM



Encuentra el camino de menor coste entre un nodo origen s y un nodo destino t.

 A diferencia del TSP:
   - No se visitan todos los nodos, solo los del camino óptimo
   - No es un ciclo, es un camino de s a t
   - No hay problema de subciclos: el flujo va en una dirección

 Variables:
   - x[i,j] ∈ {0,1}  → 1 si usamos el arco i→j
   - y[i]   ∈ {0,1}  → 1 si pasamos por el nodo i

 Restricciones:
   - Del origen sale exactamente 1 arco, no entra ninguno
   - Al destino llega exactamente 1 arco, no sale ninguno
   - Nodos intermedios: si se visita (y[i]=1), entra 1 arco y sale 1 arco
                        si no se visita (y[i]=0), no entra ni sale nada

 La variable y[i] es el truco clave: desacopla los nodos intermedios.
 Sin ella, el solver podría crear caminos fantasma que pasen por nodos
 no conectados al camino principal.

In [ ]:
using JuMP, HiGHS

n = 10
s = 1      # origen
t = n      # destino

# Matriz de costes aleatoria
import Random
Random.seed!(5)
c = rand(1:20, n, n)
for i in 1:n
    c[i,i] = 0
end

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@variable(model, y[1:n], Bin)

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# Del origen sale 1, no entra nada
@constraint(model, sum(x[s,j] for j in 1:n if j != s) == 1)
@constraint(model, sum(x[i,s] for i in 1:n if i != s) == 0)

# Al destino llega 1, no sale nada
@constraint(model, sum(x[i,t] for i in 1:n if i != t) == 1)
@constraint(model, sum(x[t,j] for j in 1:n if j != t) == 0)

# Nodos intermedios: si se visita, entra 1 y sale 1
@constraint(model, [i in 1:n; i != s && i != t],
    sum(x[i,j] for j in 1:n if j != i) == y[i])
@constraint(model, [i in 1:n; i != s && i != t],
    sum(x[j,i] for j in 1:n if j != i) == y[i])

optimize!(model)

println("Coste: ", objective_value(model))
println("Camino:")
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j")
    end
end

Coste: 6.0
Camino:
  1 → 6
  6 → 10


## TSP

El TSP en una frase: tienes N ciudades, conoces el coste de ir de cualquier ciudad a cualquier otra, y quieres encontrar el recorrido más barato que las visite todas exactamente una vez y vuelva al inicio.

El problema matemático real es este: si solo dices "de cada ciudad sale exactamente 1 flecha y llega exactamente 1 flecha", el solver puede devolverte varios ciclos pequeños (subciclos) que cumplen esa regla pero no son un tour. Toda la complejidad del TSP es añadir algo al modelo que obligue a que sea un único ciclo.


Hay varias formas. La más intuitiva para empezar es la de posiciones en el tour (modelo MTZ).
La idea es simple: asigna a cada ciudad un número que diga "en qué posición del tour la visito". Si vas de ciudad i a ciudad j, entonces la posición de j tiene que ser mayor que la de i. Así es imposible que se forme un ciclo pequeño, porque en un ciclo la posición tendría que subir y bajar a la vez.

### TSP ASIMÉTRICO (ATSP):


 Todos los modelos que hemos implementado son asimétricos porque usamos
  una matriz c[i,j] donde c[i,j] puede ser diferente de c[j,i].
  En nuestros ejemplos hemos usado matrices simétricas por simplicidad,
  pero el modelo no lo exige — basta con cambiar la matriz de costes
  por una no simétrica y el modelo funciona igual.
  El TSP simétrico sería una restricción adicional: x[i,j] == x[j,i].

### TSP Clásico

#### Modelo MTZ (de ubicacion)

El modelo MTZ resuelve el TSP añadiendo una variable de posición u[i] para
cada ciudad i > 1. Esta variable indica en qué lugar del tour se visita la
ciudad i (entre 1 y n-1). El nodo 1 actúa como ancla sin variable u,
lo que impide que se formen subciclos: cualquier ciclo que no incluya
el nodo 1 no puede asignar posiciones consistentes.

La restricción clave es:
  - u[i] - u[j] + n * x[i,j] <= n-1
- Cuando x[i,j]=1 (usamos el arco), fuerza u[j] > u[i].
- Cuando x[i,j]=0 (no usamos el arco), la restricción es siempre satisfecha
y no restringe nada.

In [ ]:
using JuMP, HiGHS

n = 5
# Matriz de costes (ejemplo pequeño)
c = [0 3 1 5 8;
     3 0 6 7 9;
     1 6 0 4 2;
     5 7 4 0 3;
     8 9 2 3 0]

model = Model(HiGHS.Optimizer)
set_silent(model)

# x[i,j] = 1 si vamos de ciudad i a ciudad j
@variable(model, x[1:n, 1:n], Bin)
# u[i] = posición de ciudad i en el tour (solo para i>1)
@variable(model, u[2:n] >= 1)

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# Cada ciudad tiene exactamente 1 salida y 1 entrada
@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n if j != i) == 1)
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n if i != j) == 1)

# Sin diagonal (no ir de i a i)
@constraint(model, [i in 1:n], x[i,i] == 0)

# MTZ: si vas de i a j, la posición de j > posición de i
@constraint(model, [i in 2:n, j in 2:n, i != j],
    u[i] - u[j] + n * x[i,j] <= n - 1)

optimize!(model)
println("Coste: ", objective_value(model))
println("Ruta:")
for i in 1:n, j in 1:n
    if value(x[i,j]) > 0.5
        println("  $i → $j")
    end
end

Coste: 16.0
Ruta:
  1 → 3
  2 → 1
  3 → 5
  4 → 2
  5 → 4


#### MODELO DE FLUJO PARA EL TSP



 Idea: mandamos un "paquete" desde el nodo 1 con n-1 unidades de flujo.
 Cada nodo que visitamos consume 1 unidad.
 Un subciclo que no incluya el nodo 1 nunca recibirá flujo → imposible.

 Variables:
   - x[i,j] ∈ {0,1}  → 1 si usamos el arco i→j
   - f[i,j] >= 0      → unidades de flujo que van por el arco i→j

 Restricciones:
   - Grado: entra y sale exactamente 1 arco por nodo (igual que MTZ)
   - f[i,j] <= (n-1)*x[i,j]: solo fluye por arcos usados
   - Nodo 1 genera n-1 unidades (una por cada ciudad a visitar)
   - Resto de nodos consume exactamente 1 unidad

 Diferencia con MTZ: MTZ asigna posiciones, flujo manda un paquete.
 Ambos bloquean subciclos pero por mecanismos distintos.
 El modelo de flujo suele ser más rápido en la práctica.

In [ ]:
# MODELO DE FLUJO PARA EL TSP
#
# Idea: mandamos un "paquete" desde el nodo 1 con n-1 unidades de flujo.
# Cada nodo visitado consume 1 unidad.
# Un subciclo sin el nodo 1 nunca recibe flujo → viola la restricción de consumo → imposible.
#
# Variables:
#   x[i,j] ∈ {0,1}  → 1 si usamos el arco i→j
#   f[i,j] >= 0      → unidades de flujo que pasan por el arco i→j

using JuMP, HiGHS

n = 5
c = [0 3 1 5 8;
     3 0 6 7 9;
     1 6 0 4 2;
     5 7 4 0 3;
     8 9 2 3 0]

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@variable(model, f[1:n, 1:n] >= 0)

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# Exactamente 1 arco sale y 1 entra por nodo
@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n if j != i) == 1)
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n if i != j) == 1)

# Solo fluye por arcos usados: si x[i,j]=0 entonces f[i,j]=0
@constraint(model, [i in 1:n, j in 1:n; i != j], f[i,j] <= (n-1) * x[i,j])

# No fluye de un nodo a si mismo
@constraint(model, [i in 1:n], f[i,i] == 0)

# Nodo 1 inyecta n-1 unidades (una por cada ciudad a visitar)
@constraint(model, sum(f[1,j] for j in 2:n) == n-1)

# Cada nodo intermedio consume 1 unidad: entra - sale = 1
@constraint(model, [i in 2:n],
    sum(f[j,i] for j in 1:n if j != i) - sum(f[i,j] for j in 1:n if j != i) == 1)

optimize!(model)
println("Coste: ", objective_value(model))
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j")
    end
end

Coste: 16.0
  1 → 2
  2 → 4
  3 → 1
  4 → 5
  5 → 3


 #### MODELO DE PRECEDENCIA PARA EL TSP



 Idea: en vez de posiciones numéricas (MTZ) o flujo, usamos variables que
 indican el orden relativo entre ciudades: "¿se visita i antes que j?"
 Los subciclos son imposibles porque violarían la antisimetría:
 en un ciclo 4→5→4 necesitarías p[4,5]=1 y p[5,4]=1 a la vez, imposible.

 Variables:
   - x[i,j] ∈ {0,1}  → 1 si usamos el arco i→j
   - p[i,j] ∈ {0,1}  → 1 si la ciudad i se visita antes que j

 Restricciones anti-subciclo:
   - Si usas el arco i→j, entonces i va antes que j: p[i,j]=1
   - Antisimetría: o i va antes que j, o j antes que i, no ambos
   - Transitividad: si i antes que j, y j antes que k, entonces i antes que k

 Comparación con otros modelos:
   MTZ       → asigna posiciones numéricas, pocas variables extra
   Flujo     → manda un paquete desde nodo 1, variables continuas
   Precedencia → orden relativo entre pares, muchas variables binarias
   En la práctica el más lento de los tres para instancias grandes.

In [ ]:
using JuMP, HiGHS

n = 5
c = [0 3 1 5 8;
     3 0 6 7 9;
     1 6 0 4 2;
     5 7 4 0 3;
     8 9 2 3 0]

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@variable(model, p[1:n, 1:n], Bin)

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# Entrada y salida exactamente 1 por nodo
@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n if j != i) == 1)
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n if i != j) == 1)

# Si usas el arco i→j, entonces i va antes que j
@constraint(model, [i in 1:n, j in 2:n; i != j], p[i,j] >= x[i,j])

# Antisimetría: o i antes que j, o j antes que i
@constraint(model, [i in 2:n, j in 2:n; i != j], p[i,j] + p[j,i] == 1)

# Transitividad: si i antes j, y j antes k, entonces i antes k
@constraint(model, [i in 2:n, j in 2:n, k in 2:n; i != j && j != k && i != k],
    p[i,j] + p[j,k] - p[i,k] <= 1)

optimize!(model)

println("Coste: ", objective_value(model))
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j")
    end
end

Coste: 16.0
  1 → 3
  2 → 1
  3 → 5
  4 → 2
  5 → 4


#### MODELO DFJ (Dantzig-Fulkerson-Johnson) PARA EL TSP



A diferencia de los modelos compactos (MTZ, flujo, precedencia), DFJ no añade
todas las restricciones anti-subciclo desde el principio. En su lugar, las va
añadiendo dinámicamente solo cuando son necesarias.

El proceso es iterativo:
  1. Resuelve el TSP solo con restricciones de grado (entra 1, sale 1)
  2. Inspecciona la solución: ¿hay subciclos?
  3. Si no hay subciclos → la solución es un tour válido, terminamos
  4. Si hay subciclos → añade una restricción SEC que prohíbe ese subciclo
     concreto y vuelve al paso 1

La restricción SEC para un subconjunto S de nodos es:
  - sum(x[i,j] para i,j en S) <= |S| - 1

Es decir: dentro del grupo S no puede haber tantos arcos como para formar un ciclo.

- Ventaja: en problemas grandes, la mayoría de subciclos posibles nunca aparecen.
DFJ solo corta los que realmente aparecen, añadiendo muy pocas restricciones.
Por eso es la base de los solvers modernos (branch & cut).

- Desventaja: requiere un bucle externo que detecte subciclos e inyecte
restricciones dinámicamente, en vez de ser un modelo que se ejecuta de una vez.

In [ ]:
using JuMP, HiGHS

n = 5
c = [0 3 1 5 8;
     3 0 6 7 9;
     1 6 0 4 2;
     5 7 4 0 3;
     8 9 2 3 0]

# Función que detecta subciclos en la solución actual
# Devuelve una lista de subconjuntos, cada uno es un ciclo
function detectar_subciclos(x_val, n)
    visitado = falses(n)
    ciclos = []
    for inicio in 1:n
        if visitado[inicio]
            continue
        end
        ciclo = [inicio]
        visitado[inicio] = true
        actual = inicio
        while true
            siguiente = findfirst(j -> j != actual && x_val[actual, j] > 0.5, 1:n)
            if siguiente === nothing || visitado[siguiente]
                break
            end
            push!(ciclo, siguiente)
            visitado[siguiente] = true
            actual = siguiente
        end
        push!(ciclos, ciclo)
    end
    return ciclos
end

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# Solo restricciones de grado
@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n if j != i) == 1)
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n if i != j) == 1)

# Bucle DFJ
iteracion = 0
while true
    optimize!(model)
    iteracion += 1
    x_val = value.(x)

    ciclos = detectar_subciclos(x_val, n)

    println("Iteración $iteracion: $(length(ciclos)) ciclo(s) encontrado(s)")

    # Si hay un único ciclo, es un tour válido
    if length(ciclos) == 1
        break
    end

    # Añadir SEC para cada subciclo detectado
    for S in ciclos
        if length(S) < n
            @constraint(model,
                sum(x[i,j] for i in S, j in S if i != j) <= length(S) - 1)
        end
    end
end

println("Coste: ", objective_value(model))
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j")
    end
end

Iteración 1: 2 ciclo(s) encontrado(s)
Iteración 2: 1 ciclo(s) encontrado(s)
Coste: 16.0
  1 → 3
  2 → 1
  3 → 5
  4 → 2
  5 → 4


#### TSP con precedencia


Variante del TSP donde ciertas ciudades deben visitarse en un orden obligatorio.
Por ejemplo: recoger un paquete en A antes de entregarlo en B, o visitar un
proveedor antes que su cliente.

Se construye sobre el modelo MTZ porque ya tiene variables de posición u[i]
que indican el orden de visita. Añadir una precedencia es tan simple como
forzar que la posición de i sea menor que la de j.

TSP CON PRECEDENCIAS SOBRE MTZ

Para forzar que la ciudad i se visite antes que j basta añadir:
  - u[i] <= u[j] - 1
  
Es decir, la posición de i debe ser estrictamente menor que la de j.
No se necesitan nuevas variables ni cambios en el resto del modelo.

In [ ]:
using JuMP, HiGHS

n = 5
c = [0 3 1 5 8;
     3 0 6 7 9;
     1 6 0 4 2;
     5 7 4 0 3;
     8 9 2 3 0]

# Pares (i,j): ciudad i debe visitarse antes que ciudad j
precedencias = [(3, 5), (2, 4)]

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@variable(model, u[2:n] >= 1)

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# Entrada y salida exactamente 1 por nodo
@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n if j != i) == 1)
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n if i != j) == 1)

# MTZ: anti-subciclo con posiciones
@constraint(model, [i in 2:n, j in 2:n; i != j],
    u[i] - u[j] + n * x[i,j] <= n-1)

# Precedencias: visitar i antes que j
for (i,j) in precedencias
    @constraint(model, u[i] <= u[j] - 1)
end

optimize!(model)

println("Coste: ", objective_value(model))
println("Ruta:")
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j")
    end
end
println("Posiciones:")
for i in 2:n
    println("  Ciudad $i → posición $(round(Int, value(u[i])))")
end

Coste: 19.0
Ruta:
  1 → 2
  2 → 3
  3 → 5
  4 → 1
  5 → 4
Posiciones:
  Ciudad 2 → posición 1
  Ciudad 3 → posición 2
  Ciudad 4 → posición 4
  Ciudad 5 → posición 3


### TSP CON VENTANAS DE TIEMPO

Variante del TSP donde cada ciudad debe visitarse dentro de un intervalo
de tiempo [a_i, b_i]. Si llegas antes de a_i, debes esperar. Si llegas
después de b_i, la solución es infactible.

Se añade una variable t[i] que representa el tiempo de llegada a la ciudad i.
La restricción de propagación de tiempo usa el mismo truco Big-M que la
actualización de carga en el problema de recogidas y entregas: solo actúa
cuando el arco está activo.

Variables adicionales al TSP:
  - t[i] >= 0  → tiempo de llegada a la ciudad i

Restricciones adicionales:
  - a[i] <= t[i] <= b[i]  (ventana de tiempo)
  - Si vas de i a j: t[j] >= t[i] + tiempo_viaje[i,j]

In [2]:
using JuMP, HiGHS

c = [0 3 1 5 8;
     3 0 6 7 9;
     1 6 0 4 2;
     5 7 4 0 3;
     8 9 2 3 0]

# Ventanas de tiempo [a_i, b_i] para cada nodo
a = [0,  2,  4,  6,  8]
b = [0, 10, 10, 15, 20]

# Tiempo de viaje igual al coste
M = sum(b)  # Big-M suficientemente grande

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@variable(model, t[1:n] >= 0)

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n if j != i) == 1)
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n if i != j) == 1)

@constraint(model, t[1] == 0)
@constraint(model, [i in 1:n], t[i] >= a[i])
@constraint(model, [i in 1:n], t[i] <= b[i])

@constraint(model, [i in 1:n, j in 1:n; i != j && j != 1],
    t[j] >= t[i] + c[i,j] - M * (1 - x[i,j]))

optimize!(model)

println("Coste: ", objective_value(model))
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j  (llegada a $j: $(round(value(t[j]), digits=1)))")
    end
end

LoadError: At In[2]:19: `@variable(model, x[1:n, 1:n], Bin)`: unexpected error parsing reference set: 1:n

### Variantes del TSP donde no se visitan todas las localizaciones

#### TSP CON BENEFICIOS (TSP with Profits)



Variante del TSP donde no es obligatorio visitar todas las ciudades.
Cada ciudad tiene un beneficio por ser visitada y hay un coste de desplazamiento.
El objetivo es decidir qué ciudades visitar y en qué orden para maximizar
la diferencia entre beneficios obtenidos y costes de desplazamiento.

Características:
  - El depósito (nodo 1) siempre se visita
  - Si visitas una ciudad, entras y sales de ella exactamente una vez
  - Si no la visitas, no entra ni sale ningún arco
  - Los subciclos se evitan con el modelo de flujo

Variables:
  - x[i,j] ∈ {0,1}  → 1 si usamos el arco i→j
  - y[i]   ∈ {0,1}  → 1 si visitamos la ciudad i
  - f[i,j] >= 0      → flujo auxiliar para evitar subciclos

Objetivo:

        maximizar sum(beneficio[i] * y[i]) - sum(coste[i,j] * x[i,j])

In [ ]:
using JuMP, HiGHS
import Random

n = 8
Random.seed!(1)

# Coordenadas aleatorias y costes euclídeos
points = [(rand(1:100), rand(1:100)) for i in 1:n]
c = [floor(Int, sqrt((points[i][1]-points[j][1])^2 +
                     (points[i][2]-points[j][2])^2))
     for i in 1:n, j in 1:n]

# Beneficios aleatorios (depósito = 0)
beneficio = [0; rand(0:100, n-1)]

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@variable(model, y[1:n], Bin)
@variable(model, f[1:n, 1:n] >= 0)

# Maximizar beneficios - costes
@objective(model, Max,
    sum(beneficio[i] * y[i] for i in 1:n) -
    sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# Depósito siempre visitado
@constraint(model, y[1] == 1)

# Grado: entra y sale exactamente y[i] arcos por nodo
@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n if j != i) == y[i])
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n if i != j) == y[j])

# Flujo: solo por arcos usados
@constraint(model, [i in 1:n, j in 1:n; i != j], f[i,j] <= (n-1) * x[i,j])
@constraint(model, [i in 1:n], f[i,i] == 0)

# Nodo 1 genera flujo proporcional a los nodos visitados
@constraint(model, sum(f[1,j] for j in 2:n) == sum(y[i] for i in 2:n))

# Cada nodo visitado consume 1 unidad de flujo
@constraint(model, [i in 2:n],
    sum(f[j,i] for j in 1:n if j != i) -
    sum(f[i,j] for j in 1:n if j != i) == y[i])

optimize!(model)

println("Beneficio neto: ", objective_value(model))
println("Ciudades visitadas:")
for i in 1:n
    if value(y[i]) > 0.5
        println("  Ciudad $i (beneficio=$(beneficio[i]))")
    end
end
println("Ruta:")
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j")
    end
end

Beneficio neto: -18.000000000000004
Ciudades visitadas:
  Ciudad 1 (beneficio=0)
  Ciudad 2 (beneficio=19)
  Ciudad 4 (beneficio=31)
  Ciudad 6 (beneficio=55)
  Ciudad 7 (beneficio=62)
  Ciudad 8 (beneficio=27)
Ruta:
  1 → 7
  2 → 4
  4 → 8
  6 → 2
  7 → 6
  8 → 1


#### ORIENTEERING PROBLEM

Variante del TSP con beneficios donde existe un presupuesto máximo de
distancia/tiempo. El objetivo es maximizar los beneficios recogidos
sin superar ese presupuesto.

Diferencia con TSP con beneficios:
  - TSP con beneficios: maximiza beneficios - costes (sin límite de distancia)
  - Orienteering: maximiza beneficios con restricción de distancia máxima

Ejemplo real: un turista con tiempo limitado que quiere visitar el máximo
número de atracciones posibles en una ciudad.

Variables:
 - x[i,j] ∈ {0,1}  → 1 si usamos el arco i→j
 - y[i]   ∈ {0,1}  → 1 si visitamos el nodo i
 - f[i,j] >= 0      → flujo auxiliar anti-subciclo

Restricciones adicionales:
  - Presupuesto: sum(c[i,j] * x[i,j]) <= Budget

In [ ]:
using JuMP, HiGHS

n = 6
c = [0 3 1 5 8 4;
     3 0 6 7 9 2;
     1 6 0 4 2 8;
     5 7 4 0 3 6;
     8 9 2 3 0 5;
     4 2 8 6 5 0]

beneficio = [0, 10, 8, 15, 12, 9]
Budget = 10

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@variable(model, y[1:n], Bin)
@variable(model, f[1:n, 1:n] >= 0)

@objective(model, Max, sum(beneficio[i] * y[i] for i in 1:n))

# Depósito siempre visitado
@constraint(model, y[1] == 1)

# Presupuesto máximo
@constraint(model, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j) <= Budget)

# Grado
@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n if j != i) == y[i])
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n if i != j) == y[j])

# Flujo anti-subciclo
@constraint(model, [i in 1:n, j in 1:n; i != j], f[i,j] <= (n-1) * x[i,j])
@constraint(model, [i in 1:n], f[i,i] == 0)
@constraint(model, sum(f[1,j] for j in 2:n) == sum(y[i] for i in 2:n))
@constraint(model, [i in 2:n],
    sum(f[j,i] for j in 1:n if j != i) -
    sum(f[i,j] for j in 1:n if j != i) == y[i])

optimize!(model)

println("Beneficio total: ", objective_value(model))
println("Coste ruta: ", sum(c[i,j] * value(x[i,j]) for i in 1:n, j in 1:n if i != j))
println("Nodos visitados:")
for i in 1:n
    if value(y[i]) > 0.5
        println("  Nodo $i (beneficio=$(beneficio[i]))")
    end
end
println("Ruta:")
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j")
    end
end

Beneficio total: 23.0
Coste ruta: 10.0
Nodos visitados:
  Nodo 1 (beneficio=0)
  Nodo 3 (beneficio=8)
  Nodo 4 (beneficio=15)
Ruta:
  1 → 3
  3 → 4
  4 → 1


#### PRIZE-COLLECTING TSP

Variante del TSP donde no es obligatorio visitar todas las ciudades,
pero por cada ciudad que NO visitas pagas una penalización.
El objetivo es minimizar coste de ruta + penalizaciones por no visitar.

Diferencia con las otras variantes:
  - TSP con beneficios: maximiza beneficios - costes
  - Orienteering: maximiza beneficios con límite de distancia
  - Prize-Collecting: minimiza costes + penalizaciones por no visitar

Ejemplo real: una empresa que tiene contratos con clientes. Si no visitas
un cliente pagas una multa. Decides qué clientes visitar según si la multa
es mayor o menor que el coste de desplazamiento.

Variables:
  - x[i,j] ∈ {0,1}  → 1 si usamos el arco i→j
  - y[i]   ∈ {0,1}  → 1 si visitamos el nodo i
  - f[i,j] >= 0      → flujo auxiliar anti-subciclo

Objetivo: minimizar coste de ruta + sum(penalización[i] * (1 - y[i]))

In [ ]:
using JuMP, HiGHS

n = 6
c = [0 3 1 5 8 4;
     3 0 6 7 9 2;
     1 6 0 4 2 8;
     5 7 4 0 3 6;
     8 9 2 3 0 5;
     4 2 8 6 5 0]

# Penalización por no visitar cada nodo (depósito = 0)
penalizacion = [0, 3, 2, 4, 3, 2]

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@variable(model, y[1:n], Bin)
@variable(model, f[1:n, 1:n] >= 0)

# Minimizar coste ruta + penalizaciones por no visitar
@objective(model, Min,
    sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j) +
    sum(penalizacion[i] * (1 - y[i]) for i in 2:n))

# Depósito siempre visitado
@constraint(model, y[1] == 1)

# Grado
@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n if j != i) == y[i])
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n if i != j) == y[j])

# Flujo anti-subciclo
@constraint(model, [i in 1:n, j in 1:n; i != j], f[i,j] <= (n-1) * x[i,j])
@constraint(model, [i in 1:n], f[i,i] == 0)
@constraint(model, sum(f[1,j] for j in 2:n) == sum(y[i] for i in 2:n))
@constraint(model, [i in 2:n],
    sum(f[j,i] for j in 1:n if j != i) -
    sum(f[i,j] for j in 1:n if j != i) == y[i])

optimize!(model)

println("Coste total: ", objective_value(model))
println("Coste ruta: ",
    sum(c[i,j] * value(x[i,j]) for i in 1:n, j in 1:n if i != j))
println("Penalizaciones: ",
    sum(penalizacion[i] * (1 - value(y[i])) for i in 2:n))
println("Nodos visitados:")
for i in 1:n
    if value(y[i]) > 0.5
        println("  Nodo $i")
    end
end
println("Nodos no visitados (penalizados):")
for i in 2:n
    if value(y[i]) < 0.5
        println("  Nodo $i (penalización=$(penalizacion[i]))")
    end
end
println("Ruta:")
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j")
    end
end

Coste total: 14.0
Coste ruta: 2.0
Penalizaciones: 12.0
Nodos visitados:
  Nodo 1
  Nodo 3
Nodos no visitados (penalizados):
  Nodo 2 (penalización=3)
  Nodo 4 (penalización=4)
  Nodo 5 (penalización=3)
  Nodo 6 (penalización=2)
Ruta:
  1 → 3
  3 → 1


#### GENERALIZED TSP

Variante del TSP donde los nodos están divididos en grupos/zonas y debes
visitar exactamente un nodo de cada zona. El tour pasa por exactamente
un representante de cada grupo.

Ejemplo real: tienes varias oficinas de correos en cada ciudad. No importa
a cuál vayas, solo debes pasar por una de cada ciudad.
Otro ejemplo: Binter Canarias aterrizando en una isla — da igual el
aeropuerto, solo debes hacer escala en esa isla.

Diferencia con TSP clásico:
  - TSP clásico: visita todos los nodos
  - Generalized TSP: visita exactamente un nodo de cada grupo

Variables:
  - x[i,j] ∈ {0,1}  → 1 si usamos el arco i→j
  - y[i]   ∈ {0,1}  → 1 si visitamos el nodo i
  - f[i,j] >= 0      → flujo auxiliar anti-subciclo

Restricciones clave:
  - De cada grupo exactamente un nodo visitado
  - Si se visita un nodo, entra 1 arco y sale 1 arco
  - Si no se visita, no entra ni sale nada

In [ ]:
using JuMP, HiGHS

n = 7
grupos = [[2,3], [4,5], [6,7]]
G = length(grupos)

c = [0 3 1 5 8 4 2;
     3 0 6 7 9 2 5;
     1 6 0 4 2 8 3;
     5 7 4 0 3 6 4;
     8 9 2 3 0 5 6;
     4 2 8 6 5 0 7;
     2 5 3 4 6 7 0]

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@variable(model, y[1:n], Bin)
@variable(model, u[1:n] >= 0)

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# Depósito siempre visitado
@constraint(model, y[1] == 1)

# Exactamente un nodo por grupo
@constraint(model, [g in 1:G], sum(y[i] for i in grupos[g]) == 1)

# Grado
@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n if j != i) == y[i])
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n if i != j) == y[j])

# MTZ anti-subciclo (solo nodos visitados)
@constraint(model, [i in 1:n, j in 2:n; i != j],
    u[i] - u[j] + n * x[i,j] <= n - 1)
@constraint(model, u[1] == 0)

optimize!(model)

println("Coste: ", objective_value(model))
println("Nodos visitados por grupo:")
for g in 1:G
    for i in grupos[g]
        if value(y[i]) > 0.5
            println("  Grupo $g → nodo $i")
        end
    end
end
println("Ruta:")
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j")
    end
end

Coste: 11.0
Nodos visitados por grupo:
  Grupo 1 → nodo 3
  Grupo 2 → nodo 5
  Grupo 3 → nodo 7
Ruta:
  1 → 7
  3 → 1
  5 → 3
  7 → 5


### Variantes del TSP con carga de producto

#### TSP con recogidas y entregas

Variante del TSP donde el vehículo recoge mercancía en ciertos nodos
y la entrega en otros. Se añaden dos restricciones respecto al TSP clásico:

  1. Precedencia: el nodo de recogida debe visitarse antes que el de entrega
     (igual que en 2.2, usando las posiciones u[i] del modelo MTZ)
  2. Capacidad: la carga del vehículo nunca puede ser negativa ni superar
     la capacidad máxima Q

La carga se modela con una variable q[i] que representa las unidades
que lleva el vehículo al salir del nodo i. Si el nodo es de recogida,
la carga sube; si es de entrega, baja.

Variables:
  - x[i,j] ∈ {0,1}  → 1 si usamos el arco i→j
  - u[i]   >= 1      → posición de la ciudad i en el tour (MTZ)
  - q[i]   >= 0      → carga del vehículo al salir del nodo i

Restricciones adicionales al TSP:
  - Si vas de i a j: q[j] = q[i] + demanda[j]  (actualización de carga)
  - 0 <= q[i] <= Q  (capacidad del vehículo)
  - u[recogida] <= u[entrega] - 1  (precedencia)

In [ ]:
using JuMP, HiGHS

n = 6
c = [0  3  1  5  8  4;
     3  0  6  7  9  2;
     1  6  0  4  2  8;
     5  7  4  0  3  6;
     8  9  2  3  0  5;
     4  2  8  6  5  0]

# Demanda por nodo: positivo=recogida, negativo=entrega, 0=depósito
demanda = [0, 3, 2, -3, -2, 0]

# Pares (recogida, entrega)
pares = [(2,4), (3,5)]

# Capacidad máxima del vehículo
Q = 5

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@variable(model, u[2:n] >= 1)
@variable(model, 0 <= q[1:n] <= Q)

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# Entrada y salida exactamente 1 por nodo
@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n if j != i) == 1)
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n if i != j) == 1)

# MTZ anti-subciclo
@constraint(model, [i in 2:n, j in 2:n; i != j],
    u[i] - u[j] + n * x[i,j] <= n-1)

# Precedencias: recoger antes de entregar
for (i,j) in pares
    @constraint(model, u[i] <= u[j] - 1)
end

# Actualización de carga: si vas de i a j, q[j] = q[i] + demanda[j]
@constraint(model, [i in 1:n, j in 1:n; i != j],
    q[j] >= q[i] + demanda[j] - Q * (1 - x[i,j]))
@constraint(model, [i in 1:n, j in 1:n; i != j],
    q[j] <= q[i] + demanda[j] + Q * (1 - x[i,j]))

optimize!(model)

println("Coste: ", objective_value(model))
println("Ruta:")
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j  (carga saliendo de $j: $(round(Int, value(q[j]))))")
    end
end

Coste: 22.0
Ruta:
  1 → 6  (carga saliendo de 6: 0)
  2 → 3  (carga saliendo de 3: 5)
  3 → 5  (carga saliendo de 5: 3)
  4 → 1  (carga saliendo de 1: 0)
  5 → 4  (carga saliendo de 4: 0)
  6 → 2  (carga saliendo de 2: 3)


#### CAPACITATED DIAL-A-RIDE PROBLEM

Variante del TSP donde hay pasajeros que deben ser recogidos en un origen
y llevados a un destino. El vehículo tiene capacidad máxima de pasajeros
simultáneos y hay ventanas de tiempo para recogida y entrega.

Es el modelo más completo porque combina tres restricciones a la vez:
  1. Precedencia: recoger al pasajero antes de llevarlo a su destino
  2. Capacidad: no superar el número máximo de pasajeros simultáneos
  3. Ventanas de tiempo: recoger y entregar dentro de [a_i, b_i]

Ejemplo real: servicio de transporte a demanda. Un minibús recoge
pasajeros en sus casas y los lleva al hospital, respetando horarios
y sin superar su capacidad.

Nodos: el depósito (nodo 1) + nodos de recogida + nodos de entrega.
Cada pasajero tiene un nodo de recogida r y un nodo de entrega d.

Variables:
  - x[i,j] ∈ {0,1}  → 1 si usamos el arco i→j
  - t[i]   >= 0      → tiempo de llegada al nodo i
  - q[i]   >= 0      → número de pasajeros en el vehículo al salir de i

Restricciones:
  - Precedencia: t[recogida] < t[entrega] para cada pasajero
  - Capacidad: q[i] <= Q en todo momento
  - Ventanas de tiempo: a[i] <= t[i] <= b[i]
  - Propagación de tiempo y carga por arcos usados

In [ ]:
using JuMP, HiGHS

# Nodo 1: depósito
# Nodos 2,3,4: recogidas (pasajeros 1,2,3)
# Nodos 5,6,7: entregas  (pasajeros 1,2,3)
n = 7
Q = 2  # capacidad máxima simultánea

# Pares (recogida, entrega) por pasajero
pares = [(2,5), (3,6), (4,7)]

c = [0 3 1 5 2 4 6;
     3 0 2 4 1 5 3;
     1 2 0 3 4 2 5;
     5 4 3 0 2 3 1;
     2 1 4 2 0 3 4;
     4 5 2 3 3 0 2;
     6 3 5 1 4 2 0]

# Ventanas de tiempo
a = [0, 1, 2, 1, 5, 6, 4]
b = [20, 8, 8, 7, 15, 15, 12]

# Demanda: +1 en recogida, -1 en entrega
demanda = [0, 1, 1, 1, -1, -1, -1]

M = 50

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@variable(model, t[1:n] >= 0)
@variable(model, 0 <= q[1:n] <= Q)

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# Entrada y salida exactamente 1 por nodo
@constraint(model, [i in 1:n], sum(x[i,j] for j in 1:n if j != i) == 1)
@constraint(model, [j in 1:n], sum(x[i,j] for i in 1:n if i != j) == 1)

# Depósito: tiempo 0, carga 0
@constraint(model, t[1] == 0)
@constraint(model, q[1] == 0)

# Ventanas de tiempo
@constraint(model, [i in 1:n], t[i] >= a[i])
@constraint(model, [i in 1:n], t[i] <= b[i])

# Propagación de tiempo
@constraint(model, [i in 1:n, j in 1:n; i != j && j != 1],
    t[j] >= t[i] + c[i,j] - M * (1 - x[i,j]))

# Precedencia: recoger antes de entregar
for (r,d) in pares
    @constraint(model, t[r] <= t[d] - 1)
end

# Actualización de carga
@constraint(model, [i in 1:n, j in 1:n; i != j && j != 1],
    q[j] >= q[i] + demanda[j] - Q * (1 - x[i,j]))
@constraint(model, [i in 1:n, j in 1:n; i != j && j != 1],
    q[j] <= q[i] + demanda[j] + Q * (1 - x[i,j]))

optimize!(model)

println("Coste: ", objective_value(model))
println("Ruta:")
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j  (t=$(round(value(t[j]),digits=1)), q=$(round(value(q[j]),digits=1)))")
    end
end

Coste: 13.0
Ruta:
  1 → 3  (t=2.0, q=1.0)
  2 → 5  (t=5.0, q=1.0)
  3 → 2  (t=4.0, q=2.0)
  4 → 7  (t=8.0, q=1.0)
  5 → 4  (t=7.0, q=2.0)
  6 → 1  (t=0.0, q=0.0)
  7 → 6  (t=15.0, q=0.0)


### RUTAS NO HAMILTONIANAS - PROBLEMA DEL CARTERO CHINO (CPP)

En el TSP cada nodo se visita exactamente una vez (ruta hamiltoniana).
En el CPP el objetivo es recorrer TODOS los arcos al menos una vez
con el mínimo coste, pudiendo pasar por el mismo nodo varias veces.

Ejemplo real: un cartero que debe recorrer todas las calles de un barrio.
Para cubrir todas las calles puede tener que pasar por la misma esquina
varias veces.

Concepto clave: grado de un nodo = número de arcos que lo tocan.
Si todos los nodos tienen grado par, existe un circuito euleriano
(recorre todos los arcos exactamente una vez sin repetir).
Si hay nodos de grado impar, hay que duplicar algunos arcos para
equilibrarlos — esos arcos duplicados son el coste extra del cartero.

El modelo busca qué arcos duplicar (añadir recorridos extra) con el
mínimo coste total, de forma que todos los nodos queden con grado par.

Variables:
  - y[i,j] ∈ {0,1,2,...}  → cuántas veces extra recorremos el arco (i,j)

Restricciones:
  - Todos los nodos deben tener grado par tras añadir los arcos extra
  - Solo se pueden duplicar arcos que existen en el grafo

In [2]:
using JuMP, HiGHS

aristas = [(1,2,3), (1,3,1), (2,3,6), (2,4,7), (3,4,4), (3,5,2), (4,5,3)]
n = 5

grado = zeros(Int, n)
for (i,j,_) in aristas
    grado[i] += 1
    grado[j] += 1
end
println("Grados: ", grado)
println("Nodos de grado impar: ", findall(g -> g % 2 != 0, grado))

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, y[i in 1:n, j in 1:n; i < j] >= 0, Int)
@variable(model, z[1:n] >= 0, Int)  # grado_total / 2

# Solo duplicar arcos existentes
for i in 1:n, j in (i+1):n
    if !any(a[1]==i && a[2]==j for a in aristas)
        @constraint(model, y[i,j] == 0)
    end
end

@objective(model, Min, sum(c * y[min(i,j), max(i,j)] for (i,j,c) in aristas))

# Grado total de cada nodo debe ser par: grado[v] + extra = 2*z[v]
for v in 1:n
    extra = sum(y[min(v,j), max(v,j)] for j in 1:n if j != v)
    @constraint(model, grado[v] + extra == 2 * z[v])
end

optimize!(model)

coste_base = sum(c for (_,_,c) in aristas)
println("Coste base: ", coste_base)
println("Coste extra: ", objective_value(model))
println("Coste total: ", coste_base + objective_value(model))
println("Arcos duplicados:")
for (i,j,c) in aristas
    v = value(y[min(i,j), max(i,j)])
    if v > 0.5
        println("  $i — $j duplicado $(round(Int,v)) vez (coste=$c)")
    end
end

Grados: [2, 3, 4, 3, 2]
Nodos de grado impar: [2, 4]
Coste base: 26
Coste extra: 7.0
Coste total: 33.0
Arcos duplicados:
  2 — 4 duplicado 1 vez (coste=7)


### Problemas con varios vehículos



#### PROBLEMA DE RUTAS CON VARIOS VEHÍCULOS (3 indices)

En el TSP un solo vehículo visita todos los clientes. En el VRP hay K
vehículos que parten del depósito (nodo 1), cada uno hace su propia ruta
y vuelve al depósito. Entre todos cubren todos los clientes.

Cada vehículo tiene una capacidad máxima Q. La suma de demandas de los
clientes en su ruta no puede superar Q.

Diferencias con TSP:
  - K vehículos hacen rutas independientes desde y hasta el depósito
  - Cada cliente es visitado exactamente una vez por un solo vehículo
  - La capacidad limita cuántos clientes puede atender cada vehículo

Variables:
 -  x[i,j,k] ∈ {0,1}  → 1 si el vehículo k usa el arco i→j
 -  q[i,k]   >= 0      → carga del vehículo k al salir del nodo i

Restricciones:
  - Cada cliente visitado exactamente una vez en total (suma sobre k)
  - Cada vehículo sale y vuelve al depósito
  - Conservación de flujo por vehículo: si k entra en i, k sale de i
  - Capacidad: la carga no supera Q en ningún momento
  - Anti-subciclo: MTZ por vehículo

In [3]:
using JuMP, HiGHS

n = 6  # nodo 1 = depósito, nodos 2..6 = clientes
K = 2  # número de vehículos
Q = 10 # capacidad máxima por vehículo

c = [0 3 1 5 8 4;
     3 0 6 7 9 2;
     1 6 0 4 2 8;
     5 7 4 0 3 6;
     8 9 2 3 0 5;
     4 2 8 6 5 0]

# Demanda de cada nodo (depósito = 0)
demanda = [0, 3, 2, 4, 3, 2]

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n, 1:K], Bin)
@variable(model, q[1:n, 1:K] >= 0)
@variable(model, u[2:n, 1:K] >= 1)

@objective(model, Min,
    sum(c[i,j] * x[i,j,k] for i in 1:n, j in 1:n, k in 1:K if i != j))

# Cada cliente visitado exactamente una vez en total
@constraint(model, [j in 2:n],
    sum(x[i,j,k] for i in 1:n, k in 1:K if i != j) == 1)

# Cada vehículo sale del depósito exactamente una vez
@constraint(model, [k in 1:K],
    sum(x[1,j,k] for j in 2:n) == 1)

# Cada vehículo vuelve al depósito exactamente una vez
@constraint(model, [k in 1:K],
    sum(x[i,1,k] for i in 2:n) == 1)

# Conservación de flujo por vehículo en nodos cliente
@constraint(model, [i in 2:n, k in 1:K],
    sum(x[i,j,k] for j in 1:n if j != i) ==
    sum(x[j,i,k] for j in 1:n if j != i))

# MTZ anti-subciclo por vehículo
@constraint(model, [i in 2:n, j in 2:n, k in 1:K; i != j],
    u[i,k] - u[j,k] + n * x[i,j,k] <= n-1)

# Capacidad: actualización de carga por vehículo
@constraint(model, [i in 1:n, j in 2:n, k in 1:K; i != j],
    q[j,k] >= q[i,k] + demanda[j] - Q * (1 - x[i,j,k]))
@constraint(model, [i in 1:n, j in 2:n, k in 1:K; i != j],
    q[j,k] <= q[i,k] + demanda[j] + Q * (1 - x[i,j,k]))

# Carga inicial del depósito = 0
@constraint(model, [k in 1:K], q[1,k] == 0)

# Carga siempre dentro de capacidad
@constraint(model, [i in 1:n, k in 1:K], q[i,k] <= Q)

optimize!(model)

println("Coste total: ", objective_value(model))
for k in 1:K
    println("Vehículo $k:")
    for i in 1:n, j in 1:n
        if i != j && value(x[i,j,k]) > 0.5
            println("  $i → $j  (carga en $j: $(round(value(q[j,k]), digits=1)))")
        end
    end
end

Coste total: 20.0
Vehículo 1:
  1 → 6  (carga en 6: 2.0)
  2 → 1  (carga en 1: 0.0)
  6 → 2  (carga en 2: 5.0)
Vehículo 2:
  1 → 3  (carga en 3: 2.0)
  3 → 5  (carga en 5: 5.0)
  4 → 1  (carga en 1: 0.0)
  5 → 4  (carga en 4: 9.0)


#### CVRP CON 2 ÍNDICES Y VARIABLES FLUJO

En vez de rastrear qué vehículo usa cada arco, solo sabemos si el arco
se usa o no. El flujo f[i,j] indica cuántos vehículos pasan por ese arco.

Ventaja: menos variables binarias (n² en vez de n²*K)
Desventaja: perdemos información sobre qué vehículo hace qué ruta

Variables:
 -  x[i,j] ∈ {0,1,...,K}  → cuántos vehículos usan el arco i→j
 -  f[i,j] >= 0            → flujo de carga por el arco i→j

In [4]:
using JuMP, HiGHS

n = 6
K = 2
Q = 10

c = [0 3 1 5 8 4;
     3 0 6 7 9 2;
     1 6 0 4 2 8;
     5 7 4 0 3 6;
     8 9 2 3 0 5;
     4 2 8 6 5 0]

demanda = [0, 3, 2, 4, 3, 2]

model = Model(HiGHS.Optimizer)
set_silent(model)

# x[i,j] ahora es entero, no binario
@variable(model, x[1:n, 1:n] >= 0, Int)
@variable(model, f[1:n, 1:n] >= 0)

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# Del depósito salen exactamente K vehículos
@constraint(model, sum(x[1,j] for j in 2:n) == K)
@constraint(model, sum(x[i,1] for i in 2:n) == K)

# Cada cliente visitado exactamente una vez
@constraint(model, [i in 2:n], sum(x[i,j] for j in 1:n if j != i) == 1)
@constraint(model, [j in 2:n], sum(x[i,j] for i in 1:n if i != j) == 1)

# Flujo: sale Q*K del depósito menos la demanda acumulada
@constraint(model, sum(f[1,j] for j in 2:n) == sum(demanda[i] for i in 2:n))

# Conservación de flujo en clientes: entra - sale = demanda
@constraint(model, [i in 2:n],
    sum(f[j,i] for j in 1:n if j != i) -
    sum(f[i,j] for j in 1:n if j != i) == demanda[i])

# Flujo acotado por capacidad y arcos usados
@constraint(model, [i in 1:n, j in 1:n; i != j],
    f[i,j] <= Q * x[i,j])

optimize!(model)

println("Coste total: ", objective_value(model))
println("Ruta:")
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j  (flujo=$(round(value(f[i,j]),digits=1)))")
    end
end

Coste total: 20.0
Ruta:
  1 → 2  (flujo=5.0)
  1 → 4  (flujo=9.0)
  2 → 6  (flujo=2.0)
  3 → 1  (flujo=0.0)
  4 → 5  (flujo=5.0)
  5 → 3  (flujo=2.0)
  6 → 1  (flujo=0.0)


#### CVRP CON 2 ÍNDICES SIN VARIABLES FLUJO

La formulación más compacta. Solo variables x[i,j] enteras.
Sin flujo auxiliar — usa generación dinámica de restricciones (DFJ)
para garantizar que las rutas son válidas.

La restricción SEC adaptada al VRP es:
  - sum(x[i,j] para i,j en S) <= |S| - 1  para todo S de clientes

Igual que DFJ en TSP pero ahora S solo contiene clientes, no el depósito.

Ventaja: muy pocas variables
Desventaja: requiere bucle iterativo como DFJ

In [ ]:
using JuMP, HiGHS

n = 6
K = 2
Q = 10

c = [0 3 1 5 8 4;
     3 0 6 7 9 2;
     1 6 0 4 2 8;
     5 7 4 0 3 6;
     8 9 2 3 0 5;
     4 2 8 6 5 0]

demanda = [0, 3, 2, 4, 3, 2]

function detectar_subciclos_clientes(x_val, n)
    visitado = falses(n)
    visitado[1] = true
    ciclos = []
    for inicio in 2:n
        if visitado[inicio]
            continue
        end
        ciclo = [inicio]
        visitado[inicio] = true
        actual = inicio
        while true
            siguiente = findfirst(j -> j != actual && x_val[actual,j] > 0.5, 2:n)
            if siguiente === nothing
                break
            end
            siguiente += 1
            if visitado[siguiente]
                break
            end
            push!(ciclo, siguiente)
            visitado[siguiente] = true
            actual = siguiente
        end
        push!(ciclos, ciclo)
    end
    return ciclos
end

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n] >= 0, Int)

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# K vehículos salen y vuelven al depósito
@constraint(model, sum(x[1,j] for j in 2:n) == K)
@constraint(model, sum(x[i,1] for i in 2:n) == K)

# Cada cliente visitado exactamente una vez
@constraint(model, [i in 2:n], sum(x[i,j] for j in 1:n if j != i) == 1)
@constraint(model, [j in 2:n], sum(x[i,j] for i in 1:n if i != j) == 1)

# Bucle DFJ
iteracion = 0
while true
    optimize!(model)
    iteracion += 1
    x_val = value.(x)
    ciclos = detectar_subciclos_clientes(x_val, n)
    println("Iteración $iteracion: $(length(ciclos)) subciclo(s)")
    if length(ciclos) == 0
        break
    end
    for S in ciclos
        @constraint(model,
            sum(x[i,j] for i in S, j in S if i != j) <= length(S) - 1)
    end
end

println("Coste total: ", objective_value(model))
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j")
    end
end

Iteración 1: 3 subciclo(s)
Iteración 2: 3 subciclo(s)
Iteración 3: 3 subciclo(s)
Iteración 4: 3 subciclo(s)
Iteración 5: 3 subciclo(s)
Iteración 6: 3 subciclo(s)
Iteración 7: 3 subciclo(s)
Iteración 8: 3 subciclo(s)
Iteración 9: 3 subciclo(s)
Iteración 10: 3 subciclo(s)
Iteración 11: 3 subciclo(s)
Iteración 12: 3 subciclo(s)
Iteración 13: 3 subciclo(s)
Iteración 14: 3 subciclo(s)
Iteración 15: 3 subciclo(s)
Iteración 16: 3 subciclo(s)
Iteración 17: 3 subciclo(s)
Iteración 18: 3 subciclo(s)
Iteración 19: 3 subciclo(s)
Iteración 20: 3 subciclo(s)
Iteración 21: 3 subciclo(s)
Iteración 22: 3 subciclo(s)
Iteración 23: 3 subciclo(s)
Iteración 24: 3 subciclo(s)
Iteración 25: 3 subciclo(s)
Iteración 26: 3 subciclo(s)
Iteración 27: 3 subciclo(s)
Iteración 28: 3 subciclo(s)
Iteración 29: 3 subciclo(s)
Iteración 30: 3 subciclo(s)
Iteración 31: 3 subciclo(s)
Iteración 32: 3 subciclo(s)
Iteración 33: 3 subciclo(s)
Iteración 34: 3 subciclo(s)
Iteración 35: 3 subciclo(s)
Iteración 36: 3 subciclo(s)
I

LoadError: Result index of attribute MathOptInterface.VariablePrimal(1) out of bounds. There are currently 0 solution(s) in the model.

#### TSP CON MÚLTIPLES DEPÓSITOS

Variante del TSP donde hay varios depósitos posibles como punto de inicio
y fin del tour. El solver elige el depósito más conveniente.

Diferencias con el TSP clásico:
  - Los depósitos no tienen restricción de visita obligatoria
  - Solo el depósito elegido tiene 1 arco de entrada y 1 de salida
  - Los depósitos inactivos no tienen ningún arco
  - Los clientes siguen visitándose exactamente una vez

Variables adicionales:
 -  y[d] ∈ {0,1}  → 1 si se usa el depósito d

Restricciones adicionales:
  - Exactamente un depósito activo
  - Del depósito activo sale 1 arco y llega 1 arco
  - De los depósitos inactivos no sale ni llega nada



In [ ]:
using JuMP, HiGHS

# Nodos 1,2 son depósitos. Nodos 3,4,5 son clientes.
n = 5
depositos = [1, 2]
clientes = [3, 4, 5]

c = [0 3 1 5 8;
     3 0 6 7 9;
     1 6 0 4 2;
     5 7 4 0 3;
     8 9 2 3 0]

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:n], Bin)
@variable(model, u[1:n] >= 0)
@variable(model, y[depositos], Bin)  # ¿se usa el depósito d?

@objective(model, Min, sum(c[i,j] * x[i,j] for i in 1:n, j in 1:n if i != j))

# Exactamente un depósito activo
@constraint(model, sum(y[d] for d in depositos) == 1)

# Del depósito activo sale 1 arco, del inactivo no sale nada
@constraint(model, [d in depositos],
    sum(x[d,j] for j in 1:n if j != d) == y[d])

# Al depósito activo llega 1 arco, al inactivo no llega nada
@constraint(model, [d in depositos],
    sum(x[i,d] for i in 1:n if i != d) == y[d])

# Cada cliente se visita exactamente una vez
@constraint(model, [i in clientes], sum(x[i,j] for j in 1:n if j != i) == 1)
@constraint(model, [j in clientes], sum(x[i,j] for i in 1:n if i != j) == 1)

# MTZ anti-subciclo (sobre todos los nodos)
@constraint(model, [i in clientes, j in clientes; i != j],
    u[i] - u[j] + n * x[i,j] <= n-1)

optimize!(model)

println("Coste: ", objective_value(model))
println("Depósito usado: ", findfirst(d -> value(y[d]) > 0.5, depositos))
for i in 1:n, j in 1:n
    if i != j && value(x[i,j]) > 0.5
        println("  $i → $j")
    end
end

Coste: 11.0
Depósito usado: 1
  1 → 3
  3 → 5
  4 → 1
  5 → 4


# Otros problemas de logística.

## Problemas de Localización

### LOCALIZACIÓN DE PLANTAS — UFLP (Tema 3.1)



Extensión del Problema de Transporte donde abrir cada almacén tiene
un COSTE FIJO, independientemente de cuánto se use. Decides qué
almacenes abrir y cómo repartir el suministro a los clientes.

Ejemplo real: tienes 5 posibles almacenes en Canarias. Alquilar cada
uno cuesta dinero fijo (aunque mandes poco producto). ¿Cuáles abres
y desde cuál sirves a cada cliente?

Diferencia clave con Problema de Transporte:
  - Transporte: todos los almacenes están activos, minimizas envíos
  - UFLP:       decides también qué almacenes abrir (coste fijo)
  - Consecuencia: ya NO existe algoritmo polinomial (NP-hard)

¿Qué significa NP-hard aquí?
  - En Transporte el solver lo resuelve en milisegundos siempre
  - En UFLP con n plantas hay 2^n combinaciones posibles de apertura
  - Con 50 plantas → 2^50 ≈ mil billones de combinaciones
  - El solver usa Branch & Bound (inteligente pero no polinomial)

¿Qué es la relajación LP?
  - Normalmente y[i] ∈ {0,1} (binaria, costoso de resolver)
  - Relajación LP: permitimos y[i] ∈ [0,1] (continua, muy rápido)
  - Si la solución relajada ya da y[i] enteros → óptimo gratis
  - En UFLP0 esto NO ocurre → necesitamos Branch & Bound

UFLP0 — un cliente puede recibir producto de VARIAS plantas

Variables:
   - x[i,j] >= 0    → unidades enviadas desde planta i al cliente j
   - y[i] ∈ {0,1}   → 1 si abrimos la planta i, 0 si no

Restricciones:
  - [R1] Σ_i x[i,j] = demand[j]          ∀j  → cubrir toda la demanda
  - [R2] Σ_j x[i,j] ≤ supply[i] · y[i]  ∀i  → no enviar si planta cerrada
  - [R3] x[i,j] ≤ demand[j] · y[i]      ∀i,j → refuerza R2, acelera solver

 R1: lo que llega al cliente j (sumando desde todas las plantas) = su demanda

 R2: si y[i]=0 → lado derecho=0 → x[i,j]=0 para todo j (planta cerrada)

 R3: redundante con R2 pero da cotas LP más ajustadas → solver más rápido

In [ ]:
using JuMP, HiGHS, Random

# ── Datos aleatorios ──────────────────────────────────────────────
n = 5    # plantas posibles
m = 8    # clientes

Random.seed!(42)
Xf, Yf = rand(0:100, n), rand(0:100, n)
Xc, Yc = rand(0:100, m), rand(0:100, m)

cost   = [round(sqrt((Xf[i]-Xc[j])^2 + (Yf[i]-Yc[j])^2), digits=1)
          for i in 1:n, j in 1:m]
fixed  = rand(20:40, n)   # coste fijo de abrir cada planta
supply = rand(15:25, n)   # capacidad de cada planta
demand = rand(1:6,  m)    # demanda de cada cliente

# Si oferta total < demanda total, ajustamos demanda proporcionalmente
if sum(supply) < sum(demand)
    p = sum(supply) / sum(demand)
    demand = floor.(Int, demand * p)
end

# ── UFLP0: cliente puede recibir de varias plantas ────────────────

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:m] >= 0)   # flujo planta i → cliente j
@variable(model, y[1:n], Bin)         # ¿abrimos planta i?

# R1: cubrir toda la demanda de cada cliente
@constraint(model, [j=1:m], sum(x[i,j] for i=1:n) == demand[j])

# R2: no enviar desde planta cerrada / respetar capacidad
@constraint(model, [i=1:n], sum(x[i,j] for j=1:m) <= supply[i] * y[i])

# R3: refuerzo — acelera solver sin cambiar el óptimo
@constraint(model, [i=1:n, j=1:m], x[i,j] <= demand[j] * y[i])

@objective(model, Min,
    sum(fixed[i] * y[i] for i=1:n) +
    sum(cost[i,j] * x[i,j] for i=1:n, j=1:m))

optimize!(model)
println("=== UFLP0 ===")
println("Coste total : ", round(objective_value(model), digits=1))
println("Plantas abiertas: ", [i for i=1:n if value(y[i]) > 0.5])
for i=1:n, j=1:m
    v = value(x[i,j])
    if v > 0.01
        println("  Planta $i → Cliente $j : $v unidades")
    end
end



### UFLP1 — cada cliente servido por UNA SOLA planta



Añade z[i,j] ∈ {0,1} → 1 si el cliente j es servido ÚNICAMENTE por planta i

Variables:
  y[i]   ∈ {0,1}  → 1 si abrimos la planta i
  z[i,j] ∈ {0,1}  → 1 si planta i sirve al cliente j (exclusivo)

Restricciones:
  [R1] Σ_i z[i,j] = 1                          ∀j   → exactamente un proveedor
  [R2] z[i,j] ≤ y[i]                           ∀i,j → solo desde planta abierta
  [R3] Σ_j demand[j]·z[i,j] ≤ supply[i]·y[i]  ∀i   → respetar capacidad

Más realista pero más difícil: la relajación LP ya NO da solución entera.

In [ ]:
using JuMP, HiGHS, Random

# ── Datos aleatorios ──────────────────────────────────────────────
n = 5    # plantas posibles
m = 8    # clientes

Random.seed!(42)
Xf, Yf = rand(0:100, n), rand(0:100, n)
Xc, Yc = rand(0:100, m), rand(0:100, m)

cost   = [round(sqrt((Xf[i]-Xc[j])^2 + (Yf[i]-Yc[j])^2), digits=1)
          for i in 1:n, j in 1:m]
fixed  = rand(20:40, n)   # coste fijo de abrir cada planta
supply = rand(15:25, n)   # capacidad de cada planta
demand = rand(1:6,  m)    # demanda de cada cliente

# Si oferta total < demanda total, ajustamos demanda proporcionalmente
if sum(supply) < sum(demand)
    p = sum(supply) / sum(demand)
    demand = floor.(Int, demand * p)
end


# ── UFLP1: cada cliente servido por UNA sola planta ───────────────

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, y[1:n], Bin)          # ¿abrimos planta i?
@variable(model, z[1:n, 1:m], Bin)     # ¿planta i sirve al cliente j?

# R1: exactamente un proveedor por cliente
@constraint(model, [j=1:m], sum(z[i,j] for i=1:n) == 1)

# R2: solo puede servir una planta abierta
@constraint(model, [i=1:n, j=1:m], z[i,j] <= y[i])

# R3: respetar capacidad de cada planta
@constraint(model, [i=1:n],
    sum(demand[j] * z[i,j] for j=1:m) <= supply[i] * y[i])

@objective(model, Min,
    sum(fixed[i] * y[i] for i=1:n) +
    sum(cost[i,j] * demand[j] * z[i,j] for i=1:n, j=1:m))

optimize!(model)
println("=== UFLP1 ===")
println("Coste total : ", round(objective_value(model), digits=1))
println("Plantas abiertas: ", [i for i=1:n if value(y[i]) > 0.5])
for i=1:n, j=1:m
    if value(z[i,j]) > 0.5
        println("  Planta $i → Cliente $j (exclusivo)")
    end
end

### Prolema p-mediana

 El problema de las p-medianas elige exactamente p plantas (sin coste fijo)
 de entre n posibles para minimizar el coste total de transporte ponderado
 por la demanda. A diferencia del UFLP, no hay coste fijo: la restricción
 es simplemente que se abran exactamente p plantas.

 Ejemplo real: Binter Canarias quiere abrir exactamente 2 almacenes de
 repuestos entre 5 posibles islas para minimizar el coste total de
 distribución a todos sus clientes.

 Variables:
   - y[i] ∈ {0,1}   → 1 si abrimos la planta i
   - x[i,j] ∈ {0,1} → 1 si el cliente j es servido desde la planta i

 Restricciones:
   - Exactamente p plantas abiertas:  ∑ y[i] = p
   - Cada cliente servido por una planta:  ∑ᵢ x[i,j] = 1  ∀j
   - Solo plantas abiertas sirven:  x[i,j] ≤ y[i]  ∀i,j

 Diferencia clave vs UFLP: no hay coste fijo fixed[i], solo se limita
 el número de plantas con ∑ y[i] = p.

In [ ]:
using JuMP, HiGHS, Random

n = 5   # plantas posibles
m = 8   # clientes
p = 2   # plantas a abrir

Random.seed!(42)
Xf, Yf = rand(0:100, n), rand(0:100, n)
Xc, Yc = rand(0:100, m), rand(0:100, m)
demand = rand(1:10, m)
cost = [round(sqrt((Xf[i]-Xc[j])^2 + (Yf[i]-Yc[j])^2), digits=1) for i in 1:n, j in 1:m]

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:n, 1:m], Bin)
@variable(model, y[1:n], Bin)

# Exactamente p plantas
@constraint(model, sum(y[i] for i in 1:n) == p)

# Cada cliente servido por exactamente una planta
@constraint(model, [j=1:m], sum(x[i,j] for i in 1:n) == 1)

# Solo plantas abiertas sirven clientes
@constraint(model, [i=1:n, j=1:m], x[i,j] <= y[i])

@objective(model, Min, sum(demand[j] * cost[i,j] * x[i,j] for i in 1:n, j in 1:m))

optimize!(model)
println("=== p-MEDIANAS (p=$p) ===")
println("Coste total: ", round(objective_value(model), digits=1))
println("Plantas abiertas: ", [i for i in 1:n if value(y[i]) > 0.5])
for j in 1:m
    planta = findfirst(i -> value(x[i,j]) > 0.5, 1:n)
    println("  Cliente $j → Planta $planta")
end

### p-centros

El problema de los p-centros elige exactamente p plantas para minimizar
 la MÁXIMA distancia entre cualquier cliente y su planta asignada
 (criterio minimax). A diferencia de p-medianas que minimiza el coste
 total, p-centros minimiza el peor caso: ningún cliente queda demasiado
 lejos de su planta.

 Ejemplo real: localizar p hospitales de emergencias para que el
 hospital más lejano a cualquier punto de la isla esté lo más cerca
 posible. Lo que importa es el caso extremo, no el promedio.

 Variables adicionales:
   - D → variable continua que representa la distancia máxima (el "peor caso")

 Restricción adicional vs p-medianas:
   - D >= cost[i,j] * x[i,j]  ∀i,j  → D es siempre >= que cualquier distancia usada

 Objetivo: min D  (en vez de min suma ponderada de costes)

In [ ]:
using JuMP, HiGHS, Random
n = 5   # plantas posibles
m = 8   # clientes
p = 2   # plantas a abrir

Random.seed!(42)
Xf, Yf = rand(0:100, n), rand(0:100, n)
Xc, Yc = rand(0:100, m), rand(0:100, m)
demand = rand(1:10, m)
cost = [round(sqrt((Xf[i]-Xc[j])^2 + (Yf[i]-Yc[j])^2), digits=1) for i in 1:n, j in 1:m]

model2 = Model(HiGHS.Optimizer)
set_silent(model2)

@variable(model2, x2[1:n, 1:m], Bin)
@variable(model2, y2[1:n], Bin)
@variable(model2, D >= 0)   # distancia máxima

@constraint(model2, sum(y2[i] for i in 1:n) == p)
@constraint(model2, [j=1:m], sum(x2[i,j] for i in 1:n) == 1)
@constraint(model2, [i=1:n, j=1:m], x2[i,j] <= y2[i])

# D >= distancia de cada asignación activa
@constraint(model2, [i=1:n, j=1:m], D >= cost[i,j] * x2[i,j])

@objective(model2, Min, D)

optimize!(model2)
println("\n=== p-CENTROS (p=$p) ===")
println("Distancia máxima: ", round(objective_value(model2), digits=1))
println("Plantas abiertas: ", [i for i in 1:n if value(y2[i]) > 0.5])
for j in 1:m
    planta = findfirst(i -> value(x2[i,j]) > 0.5, 1:n)
    println("  Cliente $j → Planta $planta (dist=$(cost[planta,j]))")
end

## Problemas de planificacion

### Lot-sizing sin capacidad

 Una fábrica planifica cuánto producir cada día durante n periodos.
 Hay una demanda d[t] cada día que hay que satisfacer. Producir
 cualquier cantidad un día tiene un coste fijo f[t] (independiente
 de la cantidad). Cada unidad producida tiene coste c[t] y almacenarla
 hasta el día siguiente cuesta h[t].

 El objetivo es decidir cuánto producir cada día para minimizar
 la suma de costes fijos + costes de producción + costes de almacenamiento.

 Variables:
   - q[t] >= 0      → cantidad producida el día t (continua)
   - I[t] >= 0      → inventario al final del día t
   - y[t] ∈ {0,1}  → 1 si producimos algo el día t

 Restricciones:
   - Balance inventario: I[t] = I[t-1] + q[t] - d[t]   ∀t
   - Solo producir si y[t]=1: q[t] <= M * y[t]   (Big-M, M = sum(d))
   - Inventarios iniciales y finales = 0

 Nota:
 - si q[t] > 0 entonces y[t] = 1 (coste fijo se activa).
 - Si y[t] = 0, Big-M fuerza q[t] = 0.

In [ ]:
using JuMP, HiGHS, Random
n_periodos = 6
d_ls = [10, 8, 15, 12, 7, 20]   # demanda por día
f_ls = [30, 25, 35, 28, 20, 32] # coste fijo de producir
c_ls = [2,  3,  2,  4,  3,  2]  # coste unitario de producción
h_ls = [1,  1,  2,  1,  1,  2]  # coste unitario de almacenamiento
M_ls = sum(d_ls)

model3 = Model(HiGHS.Optimizer)
set_silent(model3)

@variable(model3, q[1:n_periodos] >= 0)
@variable(model3, I[0:n_periodos] >= 0)
@variable(model3, y_ls[1:n_periodos], Bin)

# Inventario inicial y final = 0
@constraint(model3, I[0] == 0)
@constraint(model3, I[n_periodos] == 0)

# Balance de inventario: lo que había + lo que produzco - demanda = lo que queda
@constraint(model3, [t=1:n_periodos], I[t] == I[t-1] + q[t] - d_ls[t])

# Si no producimos (y=0), q=0. Big-M activa el coste fijo.
@constraint(model3, [t=1:n_periodos], q[t] <= M_ls * y_ls[t])

@objective(model3, Min,
    sum(f_ls[t] * y_ls[t] for t in 1:n_periodos) +
    sum(c_ls[t] * q[t]    for t in 1:n_periodos) +
    sum(h_ls[t] * I[t]    for t in 1:n_periodos))

optimize!(model3)
println("\n=== LOT-SIZING SIN CAPACIDAD ===")
println("Coste total: ", objective_value(model3))
for t in 1:n_periodos
    println("  Día $t: produce=$(round(value(q[t]),digits=1))  inventario=$(round(value(I[t]),digits=1))  activo=$(value(y_ls[t])>0.5)")
end

### Planificacion 2 productos


Una máquina produce 2 productos en n periodos. Cada día solo puede
 producir UN producto. Si cambia de producto respecto al día anterior
 incurre en un coste fijo de reconfiguración q_cambio.
 Hay demanda, coste de producción, almacenamiento e inventario inicial.

 Variables:
   - prod[i,t] >= 0       → cantidad producida del producto i en el periodo t
   - inv[i,t]  >= 0       → inventario del producto i al final del periodo t
   - z[i,t]  ∈ {0,1}      → 1 si producimos producto i en el periodo t
   - cambio[t] ∈ {0,1}    → 1 si hay cambio de producto entre t-1 y t

 Restricciones clave:
   - Solo un producto por día:  z[1,t] + z[2,t] <= 1  ∀t
   - Balance inventario por producto
   - Cambio: cambio[t] >= z[i,t] - z[i,t-1]  (detecta transición)

In [ ]:
using JuMP, HiGHS, Random
n_t = 5
n_prod = 2

# Demanda: d[i,t]
d_prod = [8 5 10 7 6;   # producto 1
          4 9  3 8 5]   # producto 2

p_cost = [3, 4]          # coste unitario producción
h_cost = [1, 2]          # coste almacenamiento
q_cambio = 20            # coste fijo de cambio de máquina
cap_prod = [15 15 15 15 15;  # capacidad producto 1
            15 15 15 15 15]  # capacidad producto 2
I0 = [5, 3]              # inventario inicial

model4 = Model(HiGHS.Optimizer)
set_silent(model4)

@variable(model4, prod_v[1:n_prod, 1:n_t] >= 0)
@variable(model4, inv_v[1:n_prod, 0:n_t] >= 0)
@variable(model4, z_v[1:n_prod, 1:n_t], Bin)
@variable(model4, cambio_v[1:n_t], Bin)

# Inventario inicial
@constraint(model4, [i=1:n_prod], inv_v[i,0] == I0[i])

# Balance inventario
@constraint(model4, [i=1:n_prod, t=1:n_t],
    inv_v[i,t] == inv_v[i,t-1] + prod_v[i,t] - d_prod[i,t])

# Solo un producto por día
@constraint(model4, [t=1:n_t], sum(z_v[i,t] for i in 1:n_prod) <= 1)

# Capacidad de producción condicionada a z
@constraint(model4, [i=1:n_prod, t=1:n_t], prod_v[i,t] <= cap_prod[i,t] * z_v[i,t])

# Detectar cambio de producto entre t-1 y t
@constraint(model4, [i=1:n_prod, t=2:n_t], cambio_v[t] >= z_v[i,t] - z_v[i,t-1])

@objective(model4, Min,
    sum(p_cost[i] * prod_v[i,t] for i in 1:n_prod, t in 1:n_t) +
    sum(h_cost[i] * inv_v[i,t]  for i in 1:n_prod, t in 1:n_t) +
    sum(q_cambio  * cambio_v[t]  for t in 2:n_t))

optimize!(model4)
println("\n=== PLANIFICACIÓN 2 PRODUCTOS ===")
println("Coste total: ", objective_value(model4))
for t in 1:n_t
    p1 = round(value(prod_v[1,t]), digits=1)
    p2 = round(value(prod_v[2,t]), digits=1)
    println("  Día $t: prod1=$p1  prod2=$p2  cambio=$(t>1 && value(cambio_v[t])>0.5)")
end


## Telecomunciaciones


### HUB LOCATION

Red de telecomunicaciones: n nodos que se comunican entre sí.
 Algunos nodos actúan como "hubs" (concentradores). El tráfico entre
 dos nodos i y j pasa por el hub de i, luego por el hub de j.
 Los hubs tienen descuento α en el tráfico entre ellos.

 Ejemplo real: aeropuertos. Un vuelo Lanzarote→Madrid pasa por
 el hub de Canarias (Las Palmas) y el hub peninsular (Madrid-Barajas).
 El tramo hub-hub tiene tarifa reducida.

 Variables:
   - z[i,k] ∈ {0,1} → 1 si el nodo i está asignado al hub k (z[k,k]=1 si k es hub)
   - y[k,m] >= 0    → flujo total entre hub k y hub m (linealización)

 Restricciones:
   - Cada nodo asignado a exactamente un hub: ∑ₖ z[i,k] = 1  ∀i
   - Solo hubs abiertos asignan nodos: z[i,k] <= z[k,k]  ∀i,k
   - Flujo entre hubs: y[k,m] >= ∑{i∈S,j∈T} d[i,j]·(z[i,k]+z[j,m]-1)

 Objetivo: min coste_fijo_hubs + coste_acceso + α·coste_entre_hubs

 NOTA: El modelo cuadrático original (z[i,k]*z[j,m] en objetivo) se
 linealiza introduciendo x[i,j,k,m] o y[k,m]. Aquí usamos y[k,m].

In [ ]:
using JuMP, HiGHS, Random
n_hub = 5
N_hub = 1:n_hub

Random.seed!(1234)
d_hub = rand(1:100, n_hub, n_hub)
c_hub = rand(1:100, n_hub, n_hub)
f_hub = rand(50000:80000, n_hub)
for i in N_hub; d_hub[i,i] = c_hub[i,i] = 0; end
c_hub = (c_hub .+ c_hub') ./ 2

O_hub = vec(sum(d_hub, dims=1))
D_hub = vec(sum(d_hub, dims=2))
α_hub = 0.2
χ_hub = 2
δ_hub = 3

model5 = Model(HiGHS.Optimizer)
set_silent(model5)

@variable(model5, z_hub[N_hub, N_hub], Bin)
@variable(model5, x_hub[N_hub, N_hub, N_hub, N_hub] >= 0)

@objective(model5, Min,
    sum(f_hub[k] * z_hub[k,k] for k in N_hub) +
    sum((χ_hub * O_hub[i] + δ_hub * D_hub[i]) * c_hub[i,k] * z_hub[i,k]
        for i in N_hub, k in N_hub) +
    sum(α_hub * d_hub[i,j] * c_hub[k,m] * x_hub[i,j,k,m]
        for i in N_hub, j in N_hub, k in N_hub, m in N_hub))

@constraint(model5, [i in N_hub], sum(z_hub[i,k] for k in N_hub) == 1)
@constraint(model5, [i in N_hub, k in N_hub], z_hub[i,k] <= z_hub[k,k])
@constraint(model5, [i in N_hub, j in N_hub, k in N_hub],
    sum(x_hub[i,j,k,m] for m in N_hub) == z_hub[i,k])
@constraint(model5, [i in N_hub, j in N_hub, m in N_hub],
    sum(x_hub[i,j,k,m] for k in N_hub) == z_hub[j,m])

optimize!(model5)
println("\n=== HUB LOCATION ===")
println("Coste total: ", round(objective_value(model5), digits=1))
println("Hubs seleccionados: ", [k for k in N_hub if value(z_hub[k,k]) > 0.5])
for i in N_hub
    hub = findfirst(k -> value(z_hub[i,k]) > 0.5, collect(N_hub))
    println("  Nodo $i → Hub $hub")
end

## Cadenas de suministro

### Mochila

Seleccionar un subconjunto de objetos que maximice el valor total
 sin superar la capacidad de la mochila.
 Ejemplo del enunciado: Luisa elige utensilios para su mochila de 30L.

 Variables:
  -  x[i] ∈ {0,1} → 1 si incluimos el objeto i

 Restricciones:
  -  ∑ volumen[i] * x[i] <= C   (capacidad)

 Objetivo: max ∑ valor[i] * x[i]

In [ ]:
using JuMP, HiGHS, Random
C_mochila = 30
volumen   = [5, 8, 3, 12, 7, 4, 10, 6, 9, 2]
valor     = [4, 7, 3,  9, 6, 3,  8, 5, 7, 2]
n_obj     = length(volumen)

model6 = Model(HiGHS.Optimizer)
set_silent(model6)

@variable(model6, x_k[1:n_obj], Bin)

@constraint(model6, sum(volumen[i] * x_k[i] for i in 1:n_obj) <= C_mochila)

@objective(model6, Max, sum(valor[i] * x_k[i] for i in 1:n_obj))

optimize!(model6)
println("\n=== MOCHILA (KNAPSACK) ===")
println("Valor máximo: ", objective_value(model6))
seleccionados = [i for i in 1:n_obj if value(x_k[i]) > 0.5]
println("Objetos seleccionados: ", seleccionados)
println("Volumen usado: ", sum(volumen[i] for i in seleccionados), " / $C_mochila")



### Bin Packing

Empaquetar n objetos en el mínimo número de cajas de capacidad C.
 Ejemplo del enunciado: Pedro guarda objetos en el mínimo número de cajas.
 Relación con CVRP: es equivalente a decidir rutas con capacidad Q
 minimizando el número de vehículos.

 Variables:
   - x[i,j] ∈ {0,1} → 1 si el objeto i va en la caja j
   - y[j]   ∈ {0,1} → 1 si la caja j se usa

 Restricciones:
   - Cada objeto en exactamente una caja: ∑ⱼ x[i,j] = 1  ∀i
   - Capacidad por caja: ∑ᵢ tamaño[i]*x[i,j] <= C*y[j]  ∀j
   - Simetría (opcional): x[i,j] <= y[j]  ∀i,j

 Objetivo: min ∑ y[j]

In [ ]:
 using JuMP, HiGHS, Random
C_caja  = 10
tamaño  = [4, 3, 5, 2, 6, 3, 4, 5, 2, 3]
n_items = length(tamaño)
n_cajas = n_items   # en el peor caso una caja por objeto

model7 = Model(HiGHS.Optimizer)
set_silent(model7)

@variable(model7, x_bp[1:n_items, 1:n_cajas], Bin)
@variable(model7, y_bp[1:n_cajas], Bin)

# Cada objeto en exactamente una caja
@constraint(model7, [i=1:n_items], sum(x_bp[i,j] for j in 1:n_cajas) == 1)

# Capacidad de cada caja
@constraint(model7, [j=1:n_cajas],
    sum(tamaño[i] * x_bp[i,j] for i in 1:n_items) <= C_caja * y_bp[j])

# Romper simetría: objeto i solo va a caja j si j está abierta
@constraint(model7, [i=1:n_items, j=1:n_cajas], x_bp[i,j] <= y_bp[j])

@objective(model7, Min, sum(y_bp[j] for j in 1:n_cajas))

optimize!(model7)
println("\n=== BIN PACKING ===")
println("Mínimo de cajas: ", Int(round(objective_value(model7))))
for j in 1:n_cajas
    if value(y_bp[j]) > 0.5
        contenido = [i for i in 1:n_items if value(x_bp[i,j]) > 0.5]
        vol = sum(tamaño[i] for i in contenido)
        println("  Caja $j: objetos $contenido  (tamaño=$vol/$C_caja)")
    end
end